<a href="https://colab.research.google.com/github/fon11731/-kratos-rebirth-quickstart/blob/main/colab/FeiyuVoice_TTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐋 FeiyuVoice · CosyVoice3 v1.0.1 文字转语音

这是**推理**入口，使用已发布的肥鱼声库。与另一份 Notebook 相互独立，不训练或修改权重，不启动公网服务，不需要挂载 Drive。

先在 **Runtime → Change runtime type → GPU** 选择可用 NVIDIA GPU，再按顺序执行。首次安装和下载需要一些时间；免费 GPU、具体型号、显存和会话时长均不保证。请及时下载结果，断开并删除运行时会丢失临时文件。

**授权和平台边界**：只处理你有权使用的文本/干净人声，不用于冒充真人或制造误导性内容。Google Colab [当前政策](https://research.google.com/colaboratory/faq.html)禁止 creating deepfakes；素材授权不自动等于平台批准。若你的用途受限，请停止并使用获准的本地环境，不绕过限制。原声素材作者为 [星落落_oi](https://space.bilibili.com/30584746/)，训练素材已获授权；[COPYRIGHT](https://github.com/Esmerubphire/astrbot_plugin_feiyu_voice/blob/main/COPYRIGHT.md)、模型 NOTICE 和上游许可分别适用。

**验证状态：真实 Colab GPU 验收尚未完成。** 本地 Linux/CUDA 结果见仓库 [colab/VALIDATION.md](https://github.com/Esmerubphire/astrbot_plugin_feiyu_voice/blob/main/colab/VALIDATION.md)，不能视为云端 PASS。


In [ ]:
#@title 1 · 检查 GPU，准备辅助脚本
import hashlib, importlib.util, pathlib, sys, urllib.request, urllib.error
ROOT = pathlib.Path("/content/feiyuvoice")
HELPERS = ROOT / "notebook_helpers"
HELPERS.mkdir(parents=True, exist_ok=True)
# Helper bytes are SHA-256 pinned. Inference source uses a separate immutable commit.
HELPER_REF = "main"
PINS = {'feiyu_colab.py': '4caf0be709826e3d9b5fcd9d663750883ca5683439b9c5d5b8b5b0347d8096e5', 'releases.json': '8bd768465277668c818facd21c5bb4b8e9af61b56407805360e6f5985aaf8f77'}
for name, expected in PINS.items():
    target = HELPERS / name
    if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
        url = f"https://raw.githubusercontent.com/Esmerubphire/astrbot_plugin_feiyu_voice/{HELPER_REF}/colab/{name}"
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                data = response.read()
        except urllib.error.URLError as exc:
            raise RuntimeError("辅助文件尚未发布或网络不可达；未推送的验收可按 colab/README.md 开发者步骤上传同版本辅助文件。") from exc
        if hashlib.sha256(data).hexdigest() != expected:
            raise RuntimeError("辅助脚本版本与 Notebook 不匹配，请重新打开同版本 Notebook。")
        temporary = target.with_suffix(target.suffix + ".download")
        temporary.write_bytes(data)
        temporary.replace(target)
spec = importlib.util.spec_from_file_location("feiyu_colab", HELPERS / "feiyu_colab.py")
fv = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fv)
fv.check_gpu()


In [ ]:
#@title 2 · 安装独立环境（可重复运行）
fv.setup("tts", ROOT)


In [ ]:
#@title 3 · 下载并校验正式发布模型
# 使用公开 Hugging Face 仓库，不需要 Token。TTS 全包约 6.94 GiB。
fv.worker("tts", ROOT, "download")


## 选择声线并输入文本

**Cute（默认）**：F6 微调 LLM ＋发布包 N01 参考。**Gentle**：原始 C3 LLM ＋ N10 参考，并非另一份微调模型，也不是只换提示词。参考音频使用发布包内的合成参考，不上传未公开素材。

沿用发布参数：seed 42、FP32、speed 1.0、非流式、原有文本前端；原生 24 kHz 输出，另保存与发布版相同的峰值保护 PCM16 WAV。没有新增降噪、变速或音高处理。支持输入中文/英文，但未充分验证所有语言和长文本。每次生成在独立进程加载模型，切换声线不会残留上一份 LLM 权重。


In [ ]:
#@title 4 · 选择声线，输入文本并生成
style = "Gentle" #@param ["Cute", "Gentle"]
text = "你好呀，我是肥鱼。今天想听我说点什么？" #@param {type:"string"}
if not 1 <= len(text.strip()) <= 2000:
    raise ValueError("请输入 1–2000 字符；初次测试建议用一句短句。")
result_wav = fv.submit("tts", ROOT, style=style.lower(), text=text)
print("输出：", result_wav)


In [ ]:
#@title 试听结果
from IPython.display import Audio, display
display(Audio(filename=str(result_wav)))
print((result_wav.parent / "result.json").read_text(encoding="utf-8"))


In [ ]:
#@title 下载 WAV（每次生成后及时保存）
from google.colab import files
files.download(str(result_wav))


## 遇到问题

- 无 GPU / CUDA kernel 失败：重新选择可用、兼容的 GPU；初始化会检查实际 CUDA 运算，不能仅靠 GPU 名称判断支持。
- 显存不足：使用更短的音频/文本，或切换显存更充足的兼容 GPU。本项目没有认证最低显存或所有 Colab GPU。
- 下载失败：检查 Hugging Face / GitHub / PyPI / PyTorch 的访问情况，然后重跑对应单元格；校验通过的模型会复用。完整初始化失败后可重跑安装；不会改动 Notebook kernel 的包。若修改了版本或源代码，使用新会话。
- 音频错误：RVC 检查上传格式、时长和解码；TTS 检查文本与所选声线。先运行最短样例，再增大输入。
- 推理日志保留明确异常；独立环境 `requirements-resolved.txt`、`diagnostic.json`、任务 `result.json` 可用于排查。分享前删除个人文本、路径或上传文件信息。

这里只提供交互式推理；没有训练、任意第三方声线、WebUI、隧道、分离伴奏或自动混音。
